# Модуль 15 — LangGraph: смена лесоруба на рельсах

Этот ноутбук — практика к лекции «LangGraph — оркестрация» (модуль 15 на сайте курса). Идея модуля: маршрут процесса задаёте вы, модель работает на своих участках маршрута. Здесь вы соберёте этот маршрут руками — от графа без единого LLM-вызова до агента с инструментами и паузой под человека.

Сквозной герой — лесоруб в мок-лесу из модуля 11.5: сетка 5×5, деревья, рюкзак, склад. За ноутбук его смена пройдёт четыре состояния: жёсткое расписание без модели → конвейер разбора заявок с LLM-узлами → агент на графе → процесс, который замирает перед спорным решением и ждёт барона.

**Карта ноутбука:**

- **Блок 1 (ядро, keyless)** — мок-лес и четыре инструмента с конвертом ошибок.
- **Блок 2 (ядро, keyless)** — кирпичи графа: State, узлы, рёбра, StateGraph.
- **Блок 3 (ядро, keyless)** — полная смена: петля с предохранителем и журналом.
- **Блок 4 (keyless-safe)** — автодискавери локальной модели (LM Studio / Ollama).
- **Блок 5 (нужен сервер)** — сортировка заявок: LLM-узлы на конвейере.
- **Блок 6 (нужен сервер)** — агент на графе: assistant, ToolNode, tools_condition.
- **Блок 7 (ядро, keyless)** — interrupt + checkpointer: человек в петле.
- **Блок 8 (ядро, keyless)** — time travel: история снимков и переигровка.
- **Блок 9** — сводка прогона.
- **Задачи** — доработайте рабочий код (Задача 3 — обязательная часть ДЗ).

`Run all` проходит целиком без единого ключа: LLM-блоки при отсутствии сервера делают мягкий пропуск, вся механика графов работает локально на чистом Python.


## Подготовка окружения

Ставим ровно четыре пакета: `langgraph` — сам фреймворк (State, узлы, рёбра, чекпоинты), `langchain-openai` — клиент `ChatOpenAI` для локального OpenAI-совместимого сервера (модули 6.1–6.2), `grandalf` — ascii-отрисовка графа в консоли, `requests` — автодискавери сервера. Полный `langchain` не нужен: LangGraph живёт отдельно — в лекции это подводный камень про «LangGraph и LangChain — не одно и то же».


In [ ]:
import sys

assert sys.version_info >= (3, 10), (
    f"нужен Python 3.10+, у вас {sys.version_info.major}.{sys.version_info.minor}. "
    "В Colab/Kaggle это уже так; локально создайте venv поновее, например: uv venv --python 3.11"
)

%pip install -q langgraph==1.2.9 langchain-openai==1.3.5 grandalf==0.8 requests

print("[ok] зависимости установлены")


In [ ]:
import importlib.metadata as importlib_metadata
import logging

# приглушаем безвредный INFO-шум HTTP-клиента (точечно, не глобально)
logging.getLogger("httpx").setLevel(logging.WARNING)

RUN_STATUS = {}   # журнал блоков для сводки в Блоке 9

for pkg in ["langgraph", "langgraph-checkpoint", "langchain-core", "langchain-openai", "grandalf"]:
    print(f"{pkg:24s} {importlib_metadata.version(pkg)}")

print()
print("[ok] окружение готово, Python", sys.version.split()[0])


## Блок 1 (ядро, keyless). Мок-лес и инструменты лесоруба

Мир — дословно из модуля 11.5: сетка 5×5, дерево на клетках (1, 2) и (3, 1), камень на (2, 4), лесоруб и склад на (0, 0), рюкзак на 5 единиц. Одно отличие: кулдаун выключен — граф в этом ноутбуке делает десятки ходов, и честные паузы растянули бы `Run all` на минуты. Остальные правила не тронуты.

Инструменты — те же четыре по одной цели, что вы спроектировали в 11.5, с тем же конвертом: успех — `{"result": ...}`, отказ — `{"error": {"code", "message"}}` с обучающей ошибкой. Одна деталь ниже отличается от 11.5 осознанно: там от кривого направления спасал `enum` в схеме инструмента, а здесь схему соберёт LangChain из сигнатуры и docstring — `enum` в ней не будет, поэтому проверка направления ушла в код.


In [ ]:
class Forest:
    """Мок-лес из модуля 11.5: сетка 5x5, лесоруб, рюкзак, склад (кулдаун здесь выключен)."""

    def __init__(self):
        self.size = 5
        self.nodes = {(1, 2): "wood", (3, 1): "wood", (2, 4): "stone"}
        self.pos = (0, 0)        # где стоит лесоруб
        self.home = (0, 0)       # клетка склада
        self.backpack = {}       # например, {"wood": 3}
        self.cap = 5             # вместимость рюкзака
        self.stock = {}          # что уже сдано на склад

    def backpack_load(self):
        return sum(self.backpack.values())


forest = Forest()


def reset_forest():
    """Свежий мир перед каждым сценарием. Инструменты ниже смотрят на глобальную forest."""
    global forest
    forest = Forest()


print("Лес собран:", forest.size, "x", forest.size,
      "| узлы:", forest.nodes, "| лесоруб и склад на", forest.home)


In [ ]:
def move(direction: str) -> dict:
    """Шаг на одну клетку: north, south, east или west."""
    steps = {"north": (0, -1), "south": (0, 1), "east": (1, 0), "west": (-1, 0)}
    if direction not in steps:
        return {"error": {"code": "unknown_direction",
                          "message": "Допустимые направления: north, south, east, west."}}
    dx, dy = steps[direction]
    x, y = forest.pos
    forest.pos = (min(max(x + dx, 0), forest.size - 1),
                  min(max(y + dy, 0), forest.size - 1))
    return {"result": {"pos": list(forest.pos)}}


def gather() -> dict:
    """Добыть один ресурс (wood или stone) с клетки, на которой стоит лесоруб."""
    node = forest.nodes.get(forest.pos)
    if node is None:
        return {"error": {"code": "no_resource_here",
                          "message": "На этой клетке нет узла — найдите его через get_map "
                                     "и подойдите move."}}
    if forest.backpack_load() >= forest.cap:
        return {"error": {"code": "inventory_full",
                          "message": f"Рюкзак полон ({forest.cap}/{forest.cap}) — вернитесь "
                                     "на склад (0, 0) и позовите deposit."}}
    forest.backpack[node] = forest.backpack.get(node, 0) + 1
    return {"result": {"gathered": node, "backpack": dict(forest.backpack)}}


def deposit() -> dict:
    """Сдать содержимое рюкзака на склад. Работает только на клетке склада (0, 0)."""
    if forest.pos != forest.home:
        return {"error": {"code": "not_at_storehouse",
                          "message": f"Склад на клетке {list(forest.home)}, а вы — на "
                                     f"{list(forest.pos)}. Дойдите move и повторите."}}
    for res, n in forest.backpack.items():
        forest.stock[res] = forest.stock.get(res, 0) + n
    deposited = dict(forest.backpack)
    forest.backpack = {}
    return {"result": {"deposited": deposited, "stock": dict(forest.stock)}}


def get_map() -> dict:
    """Карта леса: позиция лесоруба, узлы ресурсов и клетка склада. Мир не меняет."""
    return {"result": {"pos": list(forest.pos), "home": list(forest.home),
                       "nodes": [{"pos": list(p), "resource": r}
                                 for p, r in forest.nodes.items()]}}


reset_forest()
print("gather на пустой клетке ->", gather()["error"]["code"], "(обучающая ошибка на месте)")
print("get_map ->", get_map()["result"]["nodes"])
RUN_STATUS["Блок 1: мок-лес"] = "ran"


Что значит вывод. Конверт работает как в 11.5: отказ приходит с машиночитаемым `code` (по нему будет ветвиться граф) и человекочитаемым `message` (его будет читать модель в Блоке 6). Обратите внимание: `deposit` с пустым рюкзаком — не ошибка, а честный no-op (`deposited: {}`) — идемпотентность из 11.5 никуда не делась.


## Блок 2 (ядро, keyless). Кирпичи графа: State, узлы, рёбра

Самый маленький процесс из лекции — утро лесоруба: проснулся; если силы есть — рубить, нет — отдыхать. Без модели и без леса: механика графа в чистом виде.

Четыре кирпича, на которые смотреть в коде:

- **State** — `TypedDict`, общая память процесса: всё, что нужно для решений, живёт здесь.
- **Узлы** — обычные функции: приняли state, вернули словарь **обновлений** (не весь state!).
- **Условное ребро** — функция-стрелочник: читает state, возвращает **имя** следующего узла, сама ничего не меняет.
- **StateGraph** — сборщик: узлы, рёбра, `START`/`END`, `compile()`, `invoke()`.


In [ ]:
from typing import Literal
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END


class ShiftState(TypedDict):
    energy: int   # сколько сил осталось
    wood: int     # сколько дров нарублено за смену


def wake_up(state: ShiftState):
    print("--- подъём ---")
    return {}   # ничего не меняем — узел может просто делать работу


def chop(state: ShiftState):
    print("--- рубим ---")
    return {"wood": state["wood"] + 1, "energy": state["energy"] - 1}


def rest(state: ShiftState):
    print("--- отдыхаем ---")
    return {"energy": state["energy"] + 2}


def check_energy(state: ShiftState) -> Literal["chop", "rest"]:
    if state["energy"] >= 3:
        return "chop"
    return "rest"


builder = StateGraph(ShiftState)
builder.add_node("wake_up", wake_up)
builder.add_node("chop", chop)
builder.add_node("rest", rest)

builder.add_edge(START, "wake_up")                      # прямое ребро: всегда туда
builder.add_conditional_edges("wake_up", check_energy)  # условное: спросить стрелочника
builder.add_edge("chop", END)
builder.add_edge("rest", END)

morning = builder.compile()

print("бодрое утро:  ", morning.invoke({"energy": 5, "wood": 0}))
print()
print("тяжёлое утро: ", morning.invoke({"energy": 1, "wood": 0}))
RUN_STATUS["Блок 2: кирпичи графа"] = "ran"


Что значит вывод. Один и тот же граф на разных входах прошёл разными маршрутами — и решение приняла не модель, а ваша функция `check_energy` по полю state. Второе, на что смотреть: `chop` вернул только `{"wood": ..., "energy": ...}` — а в итоге оба поля на месте: LangGraph слил дельту с состоянием сам.

Бонус, за который LangGraph любят на код-ревью: схему процесса не надо рисовать руками — она выводится из кода.


In [ ]:
print(morning.get_graph().draw_ascii())
print()
print("mermaid-версия (для вставки в доку или issue):")
print(morning.get_graph().draw_mermaid())


## Блок 3 (ядро, keyless). Полная смена: петля с предохранителем

Теперь настоящий процесс над мок-лесом. Смена лесоруба — три крупных узла в петле: `scout` (выбрать ближайшее дерево по карте) → `harvest` (дойти и рубить до полного рюкзака) → `haul` (вернуться на склад и сдать) → стрелочник: цель достигнута или ходок слишком много? — конец смены; иначе — новая ходка.

Два новых приёма относительно Блока 2:

- **Reducer** — у поля `log` стоит политика слияния `operator.add`: каждый узел возвращает список из одной записи, а LangGraph **дописывает** её в журнал, не перезаписывая. Это тот же механизм, что у `add_messages` в Блоке 6, — на простом типе его видно лучше.
- **Предохранитель** — лимит ходок в стрелочнике. Ночной инцидент из лекции (сорок ходов по кругу) лечится ровно этой строкой: правило живёт в процессе, а не в голове.

И обратите внимание на `walk_to`: дойти до клетки — это вспомогательная функция внутри узла, а не узел. Узлы — это шаги процесса, за которыми вы хотите следить (чекпоинты, журнал, развилки); всё, что мельче, остаётся обычным кодом.


In [ ]:
import operator
from typing import Annotated, Optional

MAX_TRIPS = 4   # предохранитель: больше четырёх ходок за смену не бывает


class WorkdayState(TypedDict):
    goal: int                                 # сколько дров нужно сдать на склад
    trips: int                                # счётчик ходок
    target: Optional[list]                    # выбранное дерево (клетка)
    log: Annotated[list[str], operator.add]   # журнал: каждый узел дописывает, не перезаписывает


def walk_to(dest):
    """Дойти до клетки dest шагами move. Вспомогательная функция, НЕ узел графа."""
    steps = 0
    while list(forest.pos) != list(dest):
        x, y = forest.pos
        if x < dest[0]:
            d = "east"
        elif x > dest[0]:
            d = "west"
        elif y < dest[1]:
            d = "south"
        else:
            d = "north"
        move(d)
        steps += 1
    return steps


def scout(state: WorkdayState):
    """Осмотреться: выбрать ближайшее дерево по карте."""
    x, y = forest.pos
    trees = [p for p, r in forest.nodes.items() if r == "wood"]
    nearest = min(trees, key=lambda p: abs(p[0] - x) + abs(p[1] - y))
    return {"target": list(nearest), "log": [f"scout:   иду к дереву {nearest}"]}


def harvest(state: WorkdayState):
    """Дойти до цели и рубить, пока конверт не скажет остановиться."""
    steps = walk_to(state["target"])
    chopped = 0
    code = ""
    while True:
        reply = gather()
        if "error" in reply:
            code = reply["error"]["code"]   # inventory_full -> пора на склад
            break
        chopped += 1
    return {"log": [f"harvest: {steps} шагов, +{chopped} wood, стоп по коду {code}"]}


def haul(state: WorkdayState):
    """Вернуться на склад и сдать рюкзак."""
    steps = walk_to(forest.home)
    stock = deposit()["result"]["stock"]
    return {"trips": state["trips"] + 1,
            "log": [f"haul:    {steps} шагов, на складе {stock}"]}


def day_over(state: WorkdayState) -> str:
    """Стрелочник: конец смены — по цели или по предохранителю."""
    if forest.stock.get("wood", 0) >= state["goal"]:
        return "end"
    if state["trips"] >= MAX_TRIPS:
        return "end"
    return "more"


print("[ok] узлы смены готовы: scout / harvest / haul + стрелочник day_over")


In [ ]:
def build_workday():
    """Сборщик графа смены — понадобится ещё раз в Блоке 8, поэтому функция."""
    b = StateGraph(WorkdayState)
    b.add_node("scout", scout)
    b.add_node("harvest", harvest)
    b.add_node("haul", haul)

    b.add_edge(START, "scout")
    b.add_edge("scout", "harvest")
    b.add_edge("harvest", "haul")
    b.add_conditional_edges("haul", day_over, {"more": "scout", "end": END})
    return b


workday = build_workday().compile()

reset_forest()
final = workday.invoke({"goal": 8, "trips": 0, "target": None, "log": []})

print("склад:", forest.stock, "| ходок:", final["trips"])
print()
print("журнал смены (собран reducer-ом operator.add):")
for line in final["log"]:
    print(" ", line)
RUN_STATUS["Блок 3: полная смена"] = "ran"


Что значит вывод. Цель 8 дров при рюкзаке на 5 — это две ходки: петля `haul → scout` провернулась, журнал это показывает, и каждая запись в нём пришла из своего узла отдельной дельтой.

Теперь главная проверка — предохранитель. Дадим заведомо недостижимую цель: без лимита смена крутилась бы вечно (защитный `recursion_limit` самого LangGraph в конце концов уронил бы её с ошибкой `GraphRecursionError` — но аварийный тормоз не замена рабочему).


In [ ]:
reset_forest()
crazy = workday.invoke({"goal": 999, "trips": 0, "target": None, "log": []})

print("склад:", forest.stock, "| ходок:", crazy["trips"])
print("-> смену остановил предохранитель MAX_TRIPS, а не цель и не ошибка")


## Блок 4 (keyless-safe). Локальная модель: автодискавери LM Studio / Ollama

Дальше в узлы графа встаёт модель. Как и в модуле 14 — локальная, через OpenAI-совместимый сервер из модулей 6.1–6.2. Ячейка ниже сама находит сервер: сначала LM Studio (порт 1234), потом Ollama (порт 11434); адрес и модель можно навязать переменными `LOCAL_API_BASE` / `LOCAL_MODEL_ID`. Сервера нет — Блоки 5–6 мягко пропустятся, keyless-прогон останется зелёным.

Для Блока 6 модели нужен tool calling — берите instruct-модель уровня `qwen2.5:7b` и новее.


In [ ]:
import os

import requests

CANDIDATES = [
    ("LM Studio", "http://localhost:1234/v1"),
    ("Ollama", "http://localhost:11434/v1"),
]
if os.environ.get("LOCAL_API_BASE"):
    CANDIDATES.insert(0, ("LOCAL_API_BASE", os.environ["LOCAL_API_BASE"]))

LOCAL_BASE = None    # адрес найденного сервера
LOCAL_MODEL = None   # выбранная модель

for server_name, base in CANDIDATES:
    try:
        r = requests.get(f"{base}/models", timeout=2)
        r.raise_for_status()
        ids = [item["id"] for item in r.json().get("data", [])]
    except Exception:
        continue
    chat_ids = [i for i in ids if "embed" not in i.lower()]   # embedding-модели не годятся
    if chat_ids:
        LOCAL_BASE = base
        LOCAL_MODEL = os.environ.get("LOCAL_MODEL_ID", chat_ids[0])
        print(f"{server_name}: сервер найден на {base} -> модель {LOCAL_MODEL}")
        break

if LOCAL_BASE is None:
    print("Локальный сервер не найден -> Блоки 5-6 мягко пропустятся (для keyless это норма).")


In [ ]:
llm = None

if LOCAL_BASE is None:
    print("[skip] сервера нет -> llm не создаём")
else:
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(
        model=LOCAL_MODEL,
        base_url=LOCAL_BASE,
        api_key="local",     # локальный сервер ключ не проверяет
        temperature=0,
    )
    print("проверка связи:", llm.invoke("Ответь одним коротким словом: работаешь?").content[:80])


## Блок 5 (нужен сервер). Сортировка заявок: LLM-узлы на конвейере

Первый граф с моделью внутри — из лекции: у лесоруба скопились входящие заявки, среди честных — скам про «5000 золотых». Маршрут: классифицировать → развилка → отклонить | выписать наряд → доложить.

На что смотреть: модель занимает **два узла** (классификатор и автор наряда), но маршрут целиком задан рёбрами, а развилку решает ваша функция `route_request` по полю state. Это workflow с LLM-узлами — ещё не агент: модель ни разу не выбирает, что делать дальше. Реальный двойник — триаж входящих в хелпдеске: спам-фильтр, категория, черновик ответа оператору.


In [ ]:
if llm is None:
    RUN_STATUS["Блок 5: сортировка заявок"] = "skipped"
    print("[skip] Блок 5: локальный сервер не найден")
else:
    class RequestState(TypedDict):
        request: dict                 # заявка: кто прислал и что просит
        is_scam: Optional[bool]       # вердикт классификатора
        category: Optional[str]       # тип честной заявки: firewood / logs / other
        work_order: Optional[str]     # черновик наряда для честной заявки

    def classify_request(state: RequestState):
        req = state["request"]
        prompt = f"""Ты помощник лесоруба, разбираешь входящие заявки.

Заявка от {req['sender']}: {req['text']}

Первой строкой ответь ровно одним словом: SCAM или LEGIT.
Если LEGIT — второй строкой напиши категорию: firewood, logs или other."""
        reply = llm.invoke(prompt).content
        lines = [ln.strip() for ln in reply.splitlines() if ln.strip()]
        is_scam = not lines or lines[0].upper().startswith("SCAM")
        category = lines[1].lower() if not is_scam and len(lines) > 1 else None
        return {"is_scam": is_scam, "category": category}

    def reject_scam(state: RequestState):
        print(f"[отказ] {state['request']['sender']}: заявка помечена как скам")
        return {}

    def draft_order(state: RequestState):
        req = state["request"]
        prompt = f"""Ты помощник лесоруба. Составь короткий наряд на заготовку по заявке.

Заявка от {req['sender']} (категория {state['category']}): {req['text']}

Формат: 2-3 строки — что заготовить, сколько, к какому сроку. Без лишних слов."""
        return {"work_order": llm.invoke(prompt).content}

    def notify(state: RequestState):
        print("[наряд барону на подпись]")
        print(state["work_order"])
        return {}

    def route_request(state: RequestState) -> str:
        return "scam" if state["is_scam"] else "legit"

    builder = StateGraph(RequestState)
    builder.add_node("classify_request", classify_request)
    builder.add_node("reject_scam", reject_scam)
    builder.add_node("draft_order", draft_order)
    builder.add_node("notify", notify)

    builder.add_edge(START, "classify_request")
    builder.add_conditional_edges(
        "classify_request",
        route_request,
        {"scam": "reject_scam", "legit": "draft_order"},
    )
    builder.add_edge("reject_scam", END)
    builder.add_edge("draft_order", "notify")
    builder.add_edge("notify", END)

    request_graph = builder.compile()
    RUN_STATUS["Блок 5: сортировка заявок"] = "ran"
    print("[ok] граф заявок собран")


In [ ]:
if llm is None:
    print("[skip] прогоны заявок: нужен локальный сервер")
else:
    miller = {"sender": "мельник Прохор",
              "text": "Нужно двадцать вязанок дров к пятнице, плачу по прейскуранту. "
                      "Мельница у реки, привезти к воротам."}
    lottery = {"sender": "golden-lottery@underworld",
               "text": "ПОЗДРАВЛЯЕМ! Вы выиграли 5000 золотых в лотерее Подземья! "
                       "Чтобы забрать приз, пришлите свой топор и 100 монет пошлины."}

    for req in (miller, lottery):
        print("=" * 60)
        print("заявка от:", req["sender"])
        final = request_graph.invoke({"request": req, "is_scam": None,
                                      "category": None, "work_order": None})
        verdict = "скам" if final["is_scam"] else f"дело ({final['category']})"
        print("вердикт:", verdict)


Что значит вывод. Две заявки — два разных маршрута через один граф: мельник прошёл через `draft_order` и `notify` (наряд напечатан), лотерея свернула на `reject_scam` — и **второй LLM-вызов даже не случился**: лишней работы на отклонённой ветке нет по построению. Каждое решение при этом записано в state (`is_scam`, `category`) — маршрут можно восстановить по фактам, а не по логам.


## Блок 6 (нужен сервер). Агент на графе: петля ReAct явно

Возвращаем лесорубу свободу — на рельсах. Задача «добудь 3 дерева и сдай на склад» не ложится в жёсткий маршрут: ходы зависят от карты и ошибок по пути. Нужна петля «модель думает → зовёт инструмент → смотрит результат» — та самая, что в модуле 10 пряталась в `agent.run()`, а в 11.7 вы крутили её руками.

Три готовые детали LangGraph: `add_messages` — reducer, дописывающий сообщения в историю (вы уже видели его родню `operator.add` в Блоке 3); `ToolNode` — узел-исполнитель ваших функций; `tools_condition` — стрелочник «модель попросила инструмент → в tools, ответила текстом → END». Инструменты — те же четыре функции Блока 1: LangChain соберёт их контракт из сигнатур и docstring, как это делал `@tool` в smolagents.


In [ ]:
if llm is None:
    RUN_STATUS["Блок 6: агент на графе"] = "skipped"
    print("[skip] Блок 6: локальный сервер не найден")
else:
    from langchain_core.messages import AnyMessage, HumanMessage, SystemMessage
    from langgraph.graph.message import add_messages
    from langgraph.prebuilt import ToolNode, tools_condition

    class AgentState(TypedDict):
        messages: Annotated[list[AnyMessage], add_messages]

    tools = [move, gather, deposit, get_map]
    llm_with_tools = llm.bind_tools(tools)

    SYSTEM = SystemMessage(content=(
        "Ты лесоруб в мок-лесу 5x5. Действуй только через инструменты. "
        "Сначала посмотри карту get_map. Ходи move по одной клетке за вызов. "
        "Добывай gather на клетке с деревом, сдавай deposit на складе (0, 0). "
        "Ошибки приходят с кодом и подсказкой — читай message и исправляйся. "
        "Когда задача выполнена, ответь коротким отчётом без вызова инструментов."
    ))

    def assistant(state: AgentState):
        return {"messages": [llm_with_tools.invoke([SYSTEM] + state["messages"])]}

    builder = StateGraph(AgentState)
    builder.add_node("assistant", assistant)
    builder.add_node("tools", ToolNode(tools))

    builder.add_edge(START, "assistant")
    builder.add_conditional_edges("assistant", tools_condition)
    builder.add_edge("tools", "assistant")   # ребро, замыкающее петлю

    agent = builder.compile()
    RUN_STATUS["Блок 6: агент на графе"] = "ran"
    print("[ok] агент собран; петля: assistant -> tools -> assistant")


In [ ]:
if llm is None:
    print("[skip] прогон агента: нужен локальный сервер")
else:
    reset_forest()
    task = HumanMessage(content="Добудь 3 дерева и сдай их на склад. "
                                "В конце отчитайся одной строкой.")
    tool_calls = 0
    for chunk in agent.stream({"messages": [task]},
                              config={"recursion_limit": 60},
                              stream_mode="values"):
        last = chunk["messages"][-1]
        if last.type == "ai" and last.tool_calls:
            for tc in last.tool_calls:
                tool_calls += 1
                print(f"[{tool_calls:2d}] {tc['name']}({tc['args']})")
        elif last.type == "ai":
            print()
            print("отчёт лесоруба:", last.content)

    print()
    print("склад после смены:", forest.stock,
          "| вызовов инструментов:", tool_calls)


Что значит вывод. Каждая строка `[NN] tool(...)` — решение модели в развилке `tools_condition`: здесь, а не в числе LLM-вызовов, проходит граница между workflow Блока 5 и агентом. `recursion_limit: 60` — аварийный лимит супершагов на весь прогон (у «думает + зовёт» каждый виток — два шага; смена лесоруба съедает пару десятков).

Локальная модель 7B ходит неоптимально: лишний `move`, повторный `get_map`, а то и не доведёт `deposit` до самой клетки склада — и склад в конце окажется пустым при полностью исправном графе. Это нормально и поучительно: предмет Блока 6 — механика петли, а не качество агента (за него отвечают модуль 10 и дизайн инструментов из 11.5). Читайте трассу как есть. Если же строк `[NN] tool(...)` вовсе нет, а модель сразу «отчиталась» текстом — вот это симптом модели без tool calling, проверьте её в Блоке 4.


## Блок 7 (ядро, keyless). Interrupt: человек в петле

Наряд «вырубить рощу у мельницы» законен по инструментам — но после этой рощи в лесу не останется деревьев. Такое решение не должны принимать ни скрипт, ни модель: процесс обязан замереть и спросить барона — а барон ответит утром.

Пара механизмов из лекции: **checkpointer** сохраняет снимок state после каждого узла под ключом нити `thread_id`; **interrupt** внутри узла обрывает исполнение, отдаёт вопрос наружу и позволяет процессу умереть. Возобновление — отдельный `invoke` с `Command(resume=ответ)` по той же нити: значение resume становится возвратом того самого `interrupt` внутри узла.

Заметьте: во всём блоке ни одного вызова модели — пауза с человеком в середине процесса работает целиком на локальной механике.


In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command, interrupt


class FellState(TypedDict):
    grove: str        # какую рощу велено вырубить
    trees_left: int   # сколько деревьев останется в лесу после вырубки
    approved: bool
    outcome: str


def plan_felling(state: FellState):
    print(f"наряд: вырубить рощу «{state['grove']}» "
          f"(деревьев после неё останется: {state['trees_left']})")
    return {}


def confirm_felling(state: FellState):
    if state["trees_left"] == 0:      # спорный случай — будим человека
        decision = interrupt({
            "question": f"После рощи «{state['grove']}» деревьев не останется. Рубить?"
        })
        return {"approved": decision}
    return {"approved": True}         # обычный случай — едем дальше


def fell(state: FellState):
    return {"outcome": "роща вырублена, дрова на складе"}


def cancel(state: FellState):
    return {"outcome": "вырубка отменена, наряд возвращён барону"}


def route_approval(state: FellState) -> str:
    return "yes" if state["approved"] else "no"


builder = StateGraph(FellState)
builder.add_node("plan_felling", plan_felling)
builder.add_node("confirm_felling", confirm_felling)
builder.add_node("fell", fell)
builder.add_node("cancel", cancel)

builder.add_edge(START, "plan_felling")
builder.add_edge("plan_felling", "confirm_felling")
builder.add_conditional_edges("confirm_felling", route_approval,
                              {"yes": "fell", "no": "cancel"})
builder.add_edge("fell", END)
builder.add_edge("cancel", END)

felling = builder.compile(checkpointer=InMemorySaver())
print("[ok] граф вырубки собран, checkpointer подключён")


In [ ]:
config = {"configurable": {"thread_id": "felling-1"}}   # нить этого наряда

paused = felling.invoke({"grove": "у мельницы", "trees_left": 0,
                         "approved": False, "outcome": ""}, config)

question = paused["__interrupt__"][0].value["question"]
print()
print("процесс замер, вопрос человеку:", question)
print("(invoke вернулся; процесс мог бы сейчас завершиться и ожить завтра)")
print()

final = felling.invoke(Command(resume=False), config)   # утром барон запретил
print("ответ барона: нет ->", final["outcome"])


In [ ]:
# Обычный наряд проходит без паузы: interrupt в узле просто не вызывается
smooth = felling.invoke({"grove": "восточная", "trees_left": 7,
                         "approved": False, "outcome": ""},
                        {"configurable": {"thread_id": "felling-2"}})
print("обычный наряд прошёл без паузы ->", smooth["outcome"])

# И третья нить — тот же спорный случай, но барон разрешил
config3 = {"configurable": {"thread_id": "felling-3"}}
felling.invoke({"grove": "у оврага", "trees_left": 0,
                "approved": False, "outcome": ""}, config3)
approved = felling.invoke(Command(resume=True), config3)
print("ответ барона: да  ->", approved["outcome"])

RUN_STATUS["Блок 7: interrupt"] = "ran"


Что значит вывод. Три нити — три судьбы одного графа: спорный наряд с отказом, обычный без паузы, спорный с разрешением. Ключ ко всему — `thread_id`: `Command(resume=...)` находит по нему замороженный снимок, и узел `confirm_felling` доигрывается, как будто паузы не было.

Помните камень из лекции: при возобновлении узел выполняется **с начала** — код до `interrupt` отработает второй раз (наш `print` в `plan_felling` не в счёт: это другой узел, он уже завершён и в replay не попадает; а вот побочные эффекты внутри `confirm_felling` до паузы задвоились бы). Реальные двойники пары: permission gate Claude Code, кнопка «подтвердить возврат» в саппорте, ручной approve в CI.


## Блок 8 (ядро, keyless). Time travel: история снимков и переигровка

Checkpointer делает снимок после каждого супершага — значит, у процесса есть история: что процесс знал перед каждым решением. Пересоберём граф смены из Блока 3 с checkpointer-ом, прогоним — и полистаем снимки, а потом продолжим процесс с середины.

Здесь вас ждёт важная ловушка (из тех, что видно только руками): **снимки хранят state графа, но не внешний мир**. Мок-лес живёт вне state — переигровка процесса не откатывает его склад, ровно как replay процесса в проде не отменяет уже списанную монету. Смотрите на числа в выводе.


In [ ]:
workday_ckpt = build_workday().compile(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "day-1"}}

reset_forest()
workday_ckpt.invoke({"goal": 8, "trips": 0, "target": None, "log": []}, config)
print("прогон завершён, склад:", forest.stock)
print()

history = list(workday_ckpt.get_state_history(config))
print(f"снимков в истории: {len(history)} (новые первыми):")
for snap in history:
    nxt = snap.next[0] if snap.next else "-"
    print(f"  step {snap.metadata.get('step'):>2}: "
          f"дальше -> {nxt:8s} | trips={snap.values.get('trips')} "
          f"| записей в журнале: {len(snap.values.get('log', []))}")


In [ ]:
# Переигровка: берём снимок «перед второй ходкой» (следующий узел scout, одна ходка сдана)
replay_from = next(s for s in history
                   if s.next and s.next[0] == "scout" and s.values["trips"] == 1)

print("продолжаем со снимка: trips =", replay_from.values["trips"],
      "| склад в мире СЕЙЧАС:", forest.stock)
resumed = workday_ckpt.invoke(None, replay_from.config)   # None = продолжить с этого снимка

print()
print("после переигровки: trips в state =", resumed["trips"],
      "| склад в мире =", forest.stock)
print("-> state графа откатился и доигрался, а мир — нет: склад распух.")
print("   Снимки процесса != снимки мира; побочные эффекты — ваша ответственность.")

RUN_STATUS["Блок 8: time travel"] = "ran"


## Блок 9. Сводка прогона

Самопроверка перед сдачей: ядро (Блоки 1–3, 7–8) обязано быть `ran` целиком — оно не требует ни сервера, ни ключей. LLM-блоки (5–6) — `ran` при локальном сервере, `skipped` без него; для сдачи ДЗ достаточно keyless-ядра.


In [ ]:
print(f"{'блок':38s} статус")
print("-" * 50)
for name in ["Блок 1: мок-лес", "Блок 2: кирпичи графа", "Блок 3: полная смена",
             "Блок 5: сортировка заявок", "Блок 6: агент на графе",
             "Блок 7: interrupt", "Блок 8: time travel"]:
    print(f"{name:38s} {RUN_STATUS.get(name, 'не запускался')}")

core = ["Блок 1: мок-лес", "Блок 2: кирпичи графа", "Блок 3: полная смена",
        "Блок 7: interrupt", "Блок 8: time travel"]
core_ok = all(RUN_STATUS.get(n, "").startswith("ran") for n in core)
print()
print("[ok] ядро прошло целиком" if core_ok
      else "[fail] часть ядра не прошла — перезапустите Run all")


## Задачи — доработайте рабочий код

Правило прежнее: каждая задача опирается на рабочий образец выше — вы меняете работающее, а не пишете с нуля. Задача 3 — обязательная часть ДЗ; остальные по желанию. Пометка (keyless) — сервер не нужен.


### Задача 1 (keyless). Новая развилка: прихватить камень

Образец — Блок 3. На карте есть камень на клетке (2, 4), а смена его игнорирует. Добавьте в граф смены узел `grab_stone` и стрелочник после `harvest`: если рюкзак ещё не полон — зайти за камнем (одна единица), иначе — сразу на склад.

Критерий: в журнале смены появляется запись про камень, на складе в конце лежит и `wood`, и `stone`, «бесплатного» пятого слота рюкзак не выдумывает (суммарно по-прежнему не больше 5 за ходку).


### Задача 2 (keyless). Предохранитель как узел-guard

Образец — Блок 3. Сейчас лимит ходок живёт в стрелочнике `day_over` — работает, но молча. Врежьте между `haul` и `scout` узел `guard`, который пишет в журнал предупреждение, когда осталась одна ходка до лимита, и переносит решение «продолжать или стоп» к себе (стрелочник после `guard`).

Критерий: на цели 999 в журнале видно предупреждение перед остановкой; ascii-схема графа показывает новый узел. Это тот же приём «врезаться в петлю», что и в лекции про счётчик бюджета.


### Задача 3 (каркас — keyless, обязательная). Цикл writer–critic

Соберите конвейер отчёта барону из лекции: `research` (собрать факты — возьмите `forest.stock` и журнал смены из Блока 3) → `write` (составить текст отчёта) → `critic` (проверить по чек-листу) → развилка: есть замечания — назад в `write`, нет — END.

Каркас делайте на функциях-заглушках без LLM: `write` собирает отчёт строкой, `critic` — обычная функция с правилами («есть итоговое число», «есть слово „итого"», «длина не больше 400 символов»). В state держите счётчик раундов и лимит: после трёх кругов — выход с пометкой «не сошлось». Если есть локальный сервер — замените `write` и `critic` на LLM-узлы и сравните: сколько раундов понадобилось модели.

Критерий: keyless-каркас сходится не более чем за три раунда, печатает историю замечаний по раундам; лимит реально срабатывает (проверьте, ужесточив критика до невыполнимого правила).


### Задача 4 (нужен сервер). Сломайте контракт формата

Образец — Блок 5. Скопируйте `classify_request` и уберите из промпта две последние строки (про SCAM/LEGIT первой строкой). Прогоните обе заявки. Модель начнёт отвечать прозой — и парсер уведёт всё в скам (посмотрите почему: `startswith` на первой строке).

Вывод, который надо унести: маршрут графа держится на контракте формата между LLM-узлом и стрелочником. В модуле 11.5 контракт читала модель; здесь контракт модели читает ваш код. Бонус: почините парсер так, чтобы он переживал прозу (например, ищите слово во всём ответе), и сформулируйте, чем это хуже жёсткого формата.


### Задача 5 (keyless). Time travel с правкой состояния

Образец — Блок 8. У `workday_ckpt` есть не только история, но и правка: `workday_ckpt.update_state(config, {"goal": 5})` меняет state нити между шагами. Возьмите снимок перед второй ходкой (как в Блоке 8), поправьте `goal` так, чтобы вторая ходка стала последней, и доиграйте.

Критерий: продолженный процесс завершился по цели (не по предохранителю), в журнале видно ровно две ходки. Сформулируйте одним предложением: чем `update_state` опасен в проде (подсказка: аудит решений по state).


### Задача 6 (нужен сервер, advanced). Permission gate в петле агента

Соедините Блоки 6 и 7: врежьте в петлю агента узел-ворота перед сдачей на склад — если модель зовёт `deposit`, процесс сначала замирает через `interrupt` («лесоруб хочет сдать N ресурсов — разрешить?») и продолжается только после `Command(resume=True)`.

Подсказка: агенту понадобится checkpointer при `compile`, а стрелочник после `assistant` — свой (посмотрите, как устроен `tools_condition`, и добавьте третий маршрут «в ворота», когда среди tool_calls есть `deposit`). Критерий: в трассе виден вопрос ворот до фактического `deposit`; на `resume=False` склад остаётся пустым. Это ровно permission gate из модуля 11 — теперь вашими руками.


## Что дальше

Сдача ДЗ — как обычно: прогнанный ноутбук (keyless-ядро целиком `ran`, сводка Блока 9 зелёная, Задача 3 выполнена) — `[Модуль 15, ДЗ] {ссылка}` в чат курса.

- Лекция модуля 15 на сайте курса — теория к этому ноутбуку: спектр «свобода — контроль», кирпичи графа, агент как граф, interrupt.
- Квиз модуля 15 — 8 вопросов на самопроверку (страница «Квиз» рядом с лекцией на сайте курса).
- Следующий модуль по порядку части VI — 18: OpenClaw, агент, который живёт всегда включённым.
